# 03 Exploratory Data Analysis

Explore trends, distributions, segments, anomalies, and early business signals from the cleaned **Retail Store Sales** dataset.

**Key questions answered in this notebook:**
- How is revenue distributed across product categories?
- What is the impact of discounts on transaction volume and order values?
- Which channels (Online vs. In-store) perform better?
- What are the monthly revenue trends over the recorded period?

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd
import seaborn as sns

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.dpi'] = 120

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().resolve().name == 'notebooks' else Path.cwd().resolve()

In [ ]:
DATA_PATH = PROJECT_ROOT / 'data/processed/cleaned_dataset.csv'
df = pd.read_csv(DATA_PATH, parse_dates=['transaction_date'])
print(f'Shape: {df.shape}')
df.head(3)

In [ ]:
df.describe(include='all').T

## 3.1 Discount Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

labels = ['No Discount (0)', 'Discount Applied (1)']
counts = df['discount_applied'].value_counts().sort_index()

axes[0].bar(labels, counts.values, color=['#4e9af1', '#e05c5c'], edgecolor='white', width=0.5)
axes[0].set_title('Transaction Count by Discount Status')
axes[0].set_ylabel('Count')
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 200, f'{v:,}', ha='center', fontweight='bold')

axes[1].pie(counts.values, labels=labels, autopct='%1.1f%%',
            colors=['#4e9af1', '#e05c5c'], startangle=140, wedgeprops={'edgecolor': 'white'})
axes[1].set_title('Discount Rate (% of Total Transactions)')

plt.suptitle('Discount Applied Distribution', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 3.2 Total Spent Distribution — Discount vs. No Discount

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for label, color, ax in zip([0, 1], ['#4e9af1', '#e05c5c'], axes):
    subset = df[df['discount_applied'] == label]['total_spent']
    sns.histplot(subset, bins=50, kde=True, color=color, ax=ax)
    title = 'No Discount' if label == 0 else 'Discount Applied'
    ax.set_title(f'{title} Order Value')
    ax.set_xlabel('Total Spent ($)')
    ax.set_ylabel('Count')
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))

plt.suptitle('Total Spent Distribution by Discount Status', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print('\nTotal Spent Statistics by Discount Status:')
print(df.groupby('discount_applied')['total_spent'].describe().round(2))

## 3.3 Revenue by Category

In [ ]:
revenue_cat = (
    df.groupby('category')['total_spent']
    .agg(['sum', 'count'])
    .rename(columns={'sum': 'total_revenue', 'count': 'transactions'})
    .assign(avg_order_value=lambda x: (x['total_revenue'] / x['transactions']).round(2))
    .sort_values('total_revenue', ascending=False)
    .reset_index()
)

fig, ax = plt.subplots(figsize=(10, 5))
sns.barplot(data=revenue_cat, x='category', y='total_revenue',
            palette='Greens_r', ax=ax, edgecolor='white')
ax.set_title('Total Revenue by Category', fontsize=14, fontweight='bold')
ax.set_xlabel('Category')
ax.set_ylabel('Total Revenue ($)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1000:,.0f}K'))
plt.xticks(rotation=25, ha='right')
plt.tight_layout()
plt.show()

print(revenue_cat.to_string(index=False))

## 3.4 Location Performance: Online vs In-Store

In [ ]:
loc_perf = (
    df.groupby('location')['total_spent']
    .agg(['sum', 'count'])
    .rename(columns={'sum': 'revenue', 'count': 'transactions'})
    .sort_values('revenue', ascending=False)
    .reset_index()
)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
sns.barplot(data=loc_perf, x='location', y='revenue', palette='Blues_r', ax=axes[0], edgecolor='white')
axes[0].set_title('Revenue by Location', fontweight='bold')
axes[0].set_ylabel('Total Revenue ($)')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1000:,.0f}K'))

sns.barplot(data=loc_perf, x='location', y='transactions', palette='Oranges_r', ax=axes[1], edgecolor='white')
axes[1].set_title('Transactions by Location', fontweight='bold')
axes[1].set_ylabel('Count')

plt.tight_layout()
plt.show()

## 3.5 Payment Method Preferences

In [ ]:
pay_perf = df['payment_method'].value_counts().reset_index()
pay_perf.columns = ['payment_method', 'count']

fig, ax = plt.subplots(figsize=(8, 4))
sns.barplot(data=pay_perf, y='payment_method', x='count',
            palette='Purples_r', ax=ax, edgecolor='white')
ax.set_title('Transaction Volume by Payment Method', fontsize=13, fontweight='bold')
ax.set_xlabel('Transactions')
ax.set_ylabel('Payment Method')
plt.tight_layout()
plt.show()

## 3.6 Monthly Revenue Trend

In [ ]:
monthly = (
    df.groupby(['transaction_year', 'transaction_month'])['total_spent']
    .agg(['sum', 'count'])
    .rename(columns={'sum': 'revenue', 'count': 'transactions'})
    .reset_index()
)
monthly['period'] = monthly['transaction_year'].astype(str) + '-' + monthly['transaction_month'].astype(str).str.zfill(2)
monthly = monthly.sort_values('period')

fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

axes[0].plot(monthly['period'], monthly['revenue'], marker='o', color='#2ca02c')
axes[0].set_title('Monthly Revenue Trend', fontweight='bold')
axes[0].set_ylabel('Revenue ($)')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1000:,.0f}K'))

axes[1].plot(monthly['period'], monthly['transactions'], marker='o', color='#1f77b4')
axes[1].set_title('Monthly Transaction Volume', fontweight='bold')
axes[1].set_ylabel('Transactions')
axes[1].set_xlabel('Year-Month')

plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## 3.7 Order Values by Day of Week

In [ ]:
day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']

fig, ax = plt.subplots(figsize=(10, 5))
sns.boxplot(data=df, x='transaction_day_of_week', y='total_spent', order=day_order, 
            palette='Set2', ax=ax, showfliers=False)
ax.set_title('Order Value Distribution by Day of Week', fontweight='bold')
ax.set_ylabel('Total Spent ($)')
ax.set_xlabel('Day of Week')
plt.show()

## 3.8 Correlation Heatmap (Numeric Features)

In [ ]:
num_cols = ['price_per_unit', 'quantity', 'total_spent', 'discount_applied']
corr = df[num_cols].corr()

fig, ax = plt.subplots(figsize=(8, 6))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, linewidths=0.5, ax=ax, square=True)
ax.set_title('Correlation Matrix of Numeric Retail Features', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## EDA Summary — Key Observations

| # | Finding |
|---|---|
| 1 | Revenue is not perfectly uniformly distributed across categories, but some lead visibly. |
| 2 | Applying discounts correlates differently with order value size depending on quantity purchased. |
| 3 | Online and In-store share a relatively balanced split, providing steady omnichannel revenue. |
| 4 | Monthly revenue trends show specific seasonality peaks and valleys. |
| 5 | Order values remain relatively stable across different days of the week, with slight weekend variations. |

**Next step →** `04_statistical_analysis.ipynb`